<a href="https://colab.research.google.com/github/Sameer-30/PINN_Example/blob/main/Simple_Pendulum_Oscillation_PINN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1: GENERATE GROUND-TRUTH DATA (RK4)
# ══════════════════════════════════════════════════════════════════════════════

def pendulum_rk4(theta0, omega0, g, L, dt, t_end):
    n = int(t_end / dt)
    t = np.linspace(0, t_end, n + 1)
    theta = np.zeros(n + 1)
    omega = np.zeros(n + 1)
    theta[0], omega[0] = theta0, omega0

    def f(s):
        return np.array([s[1], -(g / L) * np.sin(s[0])])

    for i in range(n):
        s = np.array([theta[i], omega[i]])
        k1 = f(s)
        k2 = f(s + 0.5 * dt * k1)
        k3 = f(s + 0.5 * dt * k2)
        k4 = f(s + dt * k3)
        s = s + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)
        theta[i+1], omega[i+1] = s

    return t, theta, omega

g, L = 9.81, 1.0
theta0, omega0 = 0.5, 0.0
dt, t_end = 0.01, 10.0

t_full, theta_full, omega_full = pendulum_rk4(theta0, omega0, g, L, dt, t_end)
print(f"Ground truth: {len(t_full)} points")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 2: PREPARE DATA & COLLOCATION POINTS
# ══════════════════════════════════════════════════════════════════════════════

t_split = 6.0
split_idx = int(t_split / dt)


t_train_np = t_full[:split_idx:5]
theta_train_np = theta_full[:split_idx:5]

t_train = torch.tensor(t_train_np, dtype=torch.float32).unsqueeze(1)
theta_train = torch.tensor(theta_train_np, dtype=torch.float32).unsqueeze(1)


n_colloc = 200
t_colloc = torch.linspace(0, t_end, n_colloc, requires_grad=True).unsqueeze(1)


t_ic = torch.zeros(1, 1, requires_grad=True)

print(f"Train: {len(t_train)} pts (t=0–{t_split}s)")
print(f"Collocation: {n_colloc} pts (t=0–{t_end}s)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 3: DEFINE THE PINN
# ══════════════════════════════════════════════════════════════════════════════


class PINN(nn.Module):
    def __init__(self, n_fourier=10):
        super().__init__()

        freqs = torch.linspace(0.5, 10.0, n_fourier)
        self.register_buffer('freqs', freqs)


        input_dim = 2 * n_fourier + 1

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),  nn.Tanh(),
            nn.Linear(64, 64),         nn.Tanh(),
            nn.Linear(64, 64),         nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, t):

        features = [t]
        for freq in self.freqs:
            features.append(torch.sin(freq * t))
            features.append(torch.cos(freq * t))
        x = torch.cat(features, dim=1)
        return self.net(x)

model = PINN(n_fourier=10)
print(f"\nPINN with Fourier features")
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 4: PINN LOSS FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def physics_residual(model, t_pts, g, L):
    """Compute θ'' + (g/L)sin(θ) using automatic differentiation."""
    theta = model(t_pts)

    # θ' = dθ/dt
    theta_t = torch.autograd.grad(theta, t_pts,
        grad_outputs=torch.ones_like(theta),
        create_graph=True)[0]

    # θ'' = d²θ/dt²
    theta_tt = torch.autograd.grad(theta_t, t_pts,
        grad_outputs=torch.ones_like(theta_t),
        create_graph=True)[0]

    # ODE residual (should = 0 if physics is satisfied)
    residual = theta_tt + (g / L) * torch.sin(theta)
    return residual, theta_t


def pinn_loss(model, t_data, theta_data, t_col, t_ic, th0, om0, g, L, lam_p, lam_ic):
    # 1) Data loss
    loss_data = torch.mean((model(t_data) - theta_data)**2)

    # 2) Physics loss
    res, _ = physics_residual(model, t_col, g, L)
    loss_phys = torch.mean(res**2)

    # 3) IC loss
    th_ic = model(t_ic)
    th_t_ic = torch.autograd.grad(th_ic, t_ic,
        grad_outputs=torch.ones_like(th_ic), create_graph=True)[0]
    loss_ic = torch.mean((th_ic - th0)**2 + (th_t_ic - om0)**2)

    total = loss_data + lam_p * loss_phys + lam_ic * loss_ic
    return total, loss_data, loss_phys, loss_ic


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 5: TRAIN THE PINN
# ══════════════════════════════════════════════════════════════════════════════

lam_p  = 1.0       # physics weight
lam_ic = 10.0      # IC weight
n_adam = 4000

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_adam)

hist = {'total': [], 'data': [], 'phys': [], 'ic': []}

print(f"\n--- Phase 1: Adam ({n_adam} epochs) ---")
for ep in range(1, n_adam + 1):
    model.train()
    total, ld, lp, li = pinn_loss(
        model, t_train, theta_train, t_colloc, t_ic,
        theta0, omega0, g, L, lam_p, lam_ic
    )
    optimizer.zero_grad()
    total.backward()
    optimizer.step()
    scheduler.step()

    hist['total'].append(total.item())
    hist['data'].append(ld.item())
    hist['phys'].append(lp.item())
    hist['ic'].append(li.item())

    if ep % 1000 == 0 or ep == 1:
        print(f"  Ep {ep:5d}  Total:{total.item():.6f}  "
              f"Data:{ld.item():.6f}  Phys:{lp.item():.6f}  IC:{li.item():.6f}")

print("Done.\n")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 6: PREDICT & EVALUATE
# ══════════════════════════════════════════════════════════════════════════════

t_eval = torch.tensor(t_full, dtype=torch.float32).unsqueeze(1)
model.eval()
with torch.no_grad():
    theta_pred = model(t_eval).numpy().flatten()

# Test metrics (t = 6–10s)
th_true_test = theta_full[split_idx:]
th_pred_test = theta_pred[split_idx:]
mse_test  = np.mean((th_true_test - th_pred_test)**2)
rmse_test = np.sqrt(mse_test)
ss_res = np.sum((th_true_test - th_pred_test)**2)
ss_tot = np.sum((th_true_test - np.mean(th_true_test))**2)
r2_test = 1 - ss_res / ss_tot

# Full metrics
mse_all = np.mean((theta_full - theta_pred)**2)
r2_all  = 1 - np.sum((theta_full - theta_pred)**2) / np.sum((theta_full - np.mean(theta_full))**2)

print(f"{'='*50}")
print(f"  PINN RESULTS")
print(f"{'='*50}")
print(f"  Test Region (t = {t_split}–{t_end}s):")
print(f"    MSE  = {mse_test:.6f}")
print(f"    RMSE = {rmse_test:.6f}")
print(f"    R²   = {r2_test:.4f}")
print(f"  Full Domain:")
print(f"    R²   = {r2_all:.4f}")
print(f"{'='*50}")
print(f"  Vanilla ANN Test R² = -17.377")
print(f"  PINN Test R²        = {r2_test:.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 7: PLOTS
# ══════════════════════════════════════════════════════════════════════════════

class SimpleANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

ann = SimpleANN()
t_norm = (t_full - t_full.min()) / (t_full.max() - t_full.min())
Xa = torch.tensor(t_norm, dtype=torch.float32).unsqueeze(1)
Ya = torch.tensor(theta_full, dtype=torch.float32).unsqueeze(1)
opt_a = torch.optim.Adam(ann.parameters(), lr=0.001)
for _ in range(2000):
    l = nn.MSELoss()(ann(Xa[:split_idx]), Ya[:split_idx])
    opt_a.zero_grad(); l.backward(); opt_a.step()
ann.eval()
with torch.no_grad():
    ann_pred = ann(Xa).numpy().flatten()


# --- Fig 1: PINN Prediction ---
fig1, ax1 = plt.subplots(figsize=(8, 5))
ax1.axvspan(t_split, t_end, alpha=0.10, color='red', label='Unseen region')
ax1.plot(t_full, theta_full, color='#1E3A5F', linewidth=0.8, label='RK4 (True)')
ax1.plot(t_full, theta_pred, color='#DC2626', linewidth=0.8, linestyle='--', label='PINN')
ax1.scatter(t_train_np, theta_train_np, color='#2563EB', s=3, zorder=5,
            label=f'Train data ({len(t_train_np)} pts)', alpha=0.6)
ax1.axvline(x=t_split, color='gray', linewidth=0.5, linestyle=':')
ax1.set_xlabel('Time (s)', fontsize=15)
ax1.set_ylabel('θ (rad)', fontsize=15)
ax1.set_title('PINN Prediction vs Ground Truth', fontsize=9, fontweight='bold')
ax1.legend(fontsize=5.5, frameon=False, loc='upper right')
ax1.tick_params(labelsize=7)
ax1.grid(True, alpha=0.3, linewidth=0.5)
ax1.text(t_split+0.15, theta_full.max()*0.8, f'Test R²={r2_test:.3f}',
         fontsize=7, color='#DC2626', fontweight='bold')
fig1.tight_layout()

# --- Fig 2: Loss Components ---
fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.semilogy(hist['total'], color='black',   linewidth=0.7, label='Total')
ax2.semilogy(hist['data'],  color='#2563EB', linewidth=0.7, label='Data')
ax2.semilogy(hist['phys'],  color='#DC2626', linewidth=0.7, label='Physics')
ax2.semilogy(hist['ic'],    color='#059669', linewidth=0.7, label='IC')
ax2.set_xlabel('Epoch', fontsize=15)
ax2.set_ylabel('Loss (log)', fontsize=15)
ax2.set_title('PINN Loss Components', fontsize=9, fontweight='bold')
ax2.legend(fontsize=6, frameon=False)
ax2.tick_params(labelsize=7)
ax2.grid(True, alpha=0.3, linewidth=0.5)
fig2.tight_layout()


# --- Fig 3: ANN vs PINN Head-to-Head ---
fig3, ax3 = plt.subplots(figsize=(8, 5))
ax3.axvspan(t_split, t_end, alpha=0.08, color='red')
ax3.plot(t_full, theta_full, color='#1E3A5F', linewidth=0.9, label='True')
ax3.plot(t_full, ann_pred,   color='#F59E0B', linewidth=0.8, linestyle='--', label='ANN')
ax3.plot(t_full, theta_pred, color='#DC2626', linewidth=0.8, linestyle='--', label='PINN')
ax3.axvline(x=t_split, color='gray', linewidth=0.5, linestyle=':')
ax3.set_xlabel('Time (s)', fontsize=15)
ax3.set_ylabel('θ (rad)', fontsize=15)
ax3.set_title('ANN vs PINN: Extrapolation', fontsize=9, fontweight='bold')
ax3.legend(fontsize=6.5, frameon=False, loc='upper left')
ax3.tick_params(labelsize=7)
ax3.grid(True, alpha=0.3, linewidth=0.5)
fig3.tight_layout()


# --- Fig 4: Error Comparison ---
fig4, ax4 = plt.subplots(figsize=(8, 5))
t_test = t_full[split_idx:]
err_ann  = theta_full[split_idx:] - ann_pred[split_idx:]
err_pinn = theta_full[split_idx:] - theta_pred[split_idx:]
ax4.plot(t_test, err_ann,  color='#F59E0B', linewidth=0.7, label='ANN Error')
ax4.plot(t_test, err_pinn, color='#DC2626', linewidth=0.7, label='PINN Error')
ax4.fill_between(t_test, err_pinn, 0, alpha=0.12, color='#DC2626')
ax4.axhline(y=0, color='black', linewidth=0.4)
ax4.set_xlabel('Time (s)', fontsize=15)
ax4.set_ylabel('Error (rad)', fontsize=15)
ax4.set_title('Extrapolation Error: ANN vs PINN', fontsize=9, fontweight='bold')
ax4.legend(fontsize=6.5, frameon=False)
ax4.tick_params(labelsize=7)
ax4.grid(True, alpha=0.3, linewidth=0.5)
fig4.tight_layout()


plt.show()